# Try omni-assistant-0.6b: audio in, long text in

[Rajan98/omni-assistant-0.6b](https://huggingface.co/Rajan98/omni-assistant-0.6b) hears speech directly and answers with a
short reply plus the **tool calls** it would make on a Linux desktop. Nothing is executed here; we only look at what it decides.

Runs on a GPU (Colab: *Runtime → Change runtime type → T4 GPU*) or on CPU (slower, needs ~6 GB RAM).

In [1]:
%pip install -q "torch>=2.6" "transformers>=5.0" huggingface_hub safetensors soundfile scipy

## 1 · Load the model
Downloads the repo (~1.3 GB) and imports its standalone `omni_assistant.py`.

In [2]:
import json, sys, time
import numpy as np
from huggingface_hub import snapshot_download

root = snapshot_download("Rajan98/omni-assistant-0.6b")
sys.path.insert(0, root)
import omni_assistant as om

t = time.time()
assistant = om.load(root)            # bf16 on GPU, float32 on CPU
print(f"loaded on {assistant['device']} in {time.time() - t:.0f} s")
print("tools:", [tool["function"]["name"] for tool in assistant["cfg"]["tools"]])

def show(result, seconds):
    print("reply :", result["reply"] or "(no spoken reply)")
    for call in result["tool_calls"]:
        print("tool  :", call["name"], json.dumps(call["arguments"]))
    if not result["tool_calls"]:
        print("tool  : (none)")
    print(f"time  : {seconds:.2f} s")

def ask(messages, waveforms=None):
    t = time.time()
    result = om.respond(assistant, messages, waveforms)
    show(result, time.time() - t)
    return result

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  967MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

loaded on cuda in 25 s
tools: ['open_app', 'open_url', 'web_search', 'set_volume', 'set_brightness', 'toggle_setting', 'media_control', 'set_timer', 'get_weather', 'take_screenshot', 'type_text', 'lock_screen']


## 2 · Audio in

Set `AUDIO_PATH` to any speech recording (wav, flac, mp3 …), e.g. you saying *"open the browser and go to wikipedia"*.
In Colab you can leave it empty and upload a file when asked. The model expects 16 kHz mono; the cell converts for you.
Clips longer than 30 s are cut.

In [ ]:
import soundfile as sf
from math import gcd
from scipy.signal import resample_poly

AUDIO_PATH = ""   # e.g. "my_command.wav"

if not AUDIO_PATH:
    try:
        from google.colab import files          # Colab: pick a file from your computer
        AUDIO_PATH = next(iter(files.upload()))
    except ImportError:
        raise SystemExit("Set AUDIO_PATH to a recording first.")

def load_16k(path):
    wave, rate = sf.read(path, dtype="float32", always_2d=True)
    wave = wave.mean(axis=1)                                     # stereo → mono
    g = gcd(rate, 16000)
    return wave if rate == 16000 else resample_poly(wave, 16000 // g, rate // g).astype(np.float32)

wave = load_16k(AUDIO_PATH)
print(f"{AUDIO_PATH}: {len(wave) / 16000:.1f} s of audio")
heard = ask([{"role": "user", "content": om.AUDIO}], [wave])

**Follow-up turn.** If the model asked a question (e.g. *"Should I type …?"* or *"Which app?"*), answer it by text
while keeping the audio turn in the conversation:

In [ ]:
if heard["reply"].rstrip().endswith("?") and not heard["tool_calls"]:
    ask([{"role": "user", "content": om.AUDIO}, om.assistant_message(heard), {"role": "user", "content": "yes, do it"}], [wave])
else:
    print("No open question; nothing to answer.")

## 3 · One long text

The model was trained on short spoken commands, so a long rambling message is a good stress test:
does it find the real requests inside the story, keep them in order, and ignore the chit-chat?
Edit `LONG_TEXT` freely.

In [3]:
LONG_TEXT = """
Okay so here's the situation. I just got home after a really long day, my manager kept moving the deadline for the
quarterly report and honestly I'm exhausted, and the room is way too bright because the afternoon sun is hitting
the screen. Anyway, my sister is coming over later and she asked me to look up a good lasagna recipe because we're
cooking together tonight, and I always forget how long the oven part takes. Oh, and whatever music is playing right
now is way too loud, I can barely think. Could you turn the volume down a bit, find me a vegetarian lasagna recipe,
and set a timer for forty five minutes so I remember to start cooking before she gets here? Thanks, you're a lifesaver.
"""
print(f"{len(LONG_TEXT.split())} words")
long_result = ask([{"role": "user", "content": " ".join(LONG_TEXT.split())}])

128 words
reply : Volume down, searching lasagna recipes, and a forty five minute timer is set.
tool  : set_volume {"action": "down"}
tool  : web_search {"query": "vegetarian lasagna recipe"}
tool  : set_timer {"seconds": 2700, "label": "cooking"}
time  : 6.45 s


## 4 · (Optional) A tool result going back

Some tools return data. Here the model asks for the weather, we pretend the desktop answered, and the model
turns that result into a short spoken summary.

In [ ]:
messages = [{"role": "user", "content": "what's the weather like tomorrow in paris"}]
first = ask(messages)
if first["tool_calls"]:
    fake = {"ok": True, "location": "Paris", "day": "tomorrow", "temp_c": 14, "condition": "light rain"}
    messages += [om.assistant_message(first), {"role": "tool", "content": json.dumps(fake)}]
    ask(messages)

### What to expect
- Typed commands: ~81% of tool calls exactly right on the held-out test set; spoken (clean synthetic voices): ~72%.
- Typical misses: free-text words it mishears (search terms, site names), or a clarifying question instead of an action.
- Real microphones and accents are harder than the synthetic test voices. See the
  [model card](https://huggingface.co/Rajan98/omni-assistant-0.6b) for details.